INGESTION DOC WORD

Docx2txtLOader


In [13]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("../data/word_files/Test.docx")
documents = loader.load()

print("nombre de documents:", len(documents))
print("contenu du document:", documents[:300])
print(documents[0].metadata)

nombre de documents: 1
contenu du document: [Document(metadata={'source': '../data/word_files/Test.docx'}, page_content='Der letzte Zug\n\nEs war ein kalter Novemberabend, als Daniel zum ersten Mal den alten Bahnhof bemerkte. Er war spät von der Universität gekommen und wollte nur noch nach Hause. Der Bahnhof lag etwas außerhalb der Stadt und war um diese Uhrzeit fast leer. Nur eine alte Laterne flackerte über dem Bahnsteig.\n\nDaniel schaute auf die Anzeigetafel. Sein Zug sollte eigentlich um 22:47 Uhr kommen. Doch plötzlich verschwand die Anzeige. Stattdessen erschien eine einzige Zeile:\n\n„Letzter Zug – 23:00 Uhr.“\n\nDaniel runzelte die Stirn. Er kannte diese Verbindung nicht. Trotzdem blieb er stehen.\n\nUm 23:00 Uhr hörte er tatsächlich einen Zug kommen. Er war alt und dunkel, ganz anders als die modernen Züge, die normalerweise an diesem Bahnhof hielten. Die Türen öffneten sich langsam.\n\nDaniel zögerte.\n\n„Steigen Sie ein“, sagte plötzlich eine Stimme hinter ihm.\n\nEr dreht

version avancéé:quand Docx2txtLoader ne suffit pas

In [14]:
from langchain_community.document_loaders import UnstructuredAPIFileIOLoader
loader = Docx2txtLoader("../data/word_files/Test.docx")
documents = loader.load()

print("nombre de documents:", len(documents))
print("contenu du document:", documents[:300])
print(documents[0].metadata)

nombre de documents: 1
contenu du document: [Document(metadata={'source': '../data/word_files/Test.docx'}, page_content='Der letzte Zug\n\nEs war ein kalter Novemberabend, als Daniel zum ersten Mal den alten Bahnhof bemerkte. Er war spät von der Universität gekommen und wollte nur noch nach Hause. Der Bahnhof lag etwas außerhalb der Stadt und war um diese Uhrzeit fast leer. Nur eine alte Laterne flackerte über dem Bahnsteig.\n\nDaniel schaute auf die Anzeigetafel. Sein Zug sollte eigentlich um 22:47 Uhr kommen. Doch plötzlich verschwand die Anzeige. Stattdessen erschien eine einzige Zeile:\n\n„Letzter Zug – 23:00 Uhr.“\n\nDaniel runzelte die Stirn. Er kannte diese Verbindung nicht. Trotzdem blieb er stehen.\n\nUm 23:00 Uhr hörte er tatsächlich einen Zug kommen. Er war alt und dunkel, ganz anders als die modernen Züge, die normalerweise an diesem Bahnhof hielten. Die Türen öffneten sich langsam.\n\nDaniel zögerte.\n\n„Steigen Sie ein“, sagte plötzlich eine Stimme hinter ihm.\n\nEr dreht

# Atelier de netoyage:Nettoyage intelligent

In [15]:
import re

def clean_word_text(text: str) -> str:
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Pipe line complete Word+clean+ split

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_word_files(file_paths):
    # chargee le document word
    docs= Docx2txtLoader(file_paths).load()
    # netoyer le texte
    cleaned = []
    for doc in docs:
        cleaned_text = clean_word_text(doc.page_content)
        cleaned.append(cleaned_text)
    # diviser le texte en morceaux
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.create_documents(cleaned)
    return chunks
    

In [17]:
chunks = process_word_files("../data/word_files/Test.docx")
print("le nombre de chunks:", len(chunks))
print(chunks[1].page_content)
print(chunks[1].metadata)

le nombre de chunks: 6
, sagte pl tzlich eine Stimme hinter ihm. Er drehte sich um. Ein lterer Mann stand dort. Er trug einen langen schwarzen Mantel und hielt eine alte Taschenuhr in der Hand. Wohin f hrt dieser Zug? , fragte Daniel. Der Mann l chelte. Dorthin, wo Sie eigentlich sein sollten. Daniel verstand die Antwort nicht. Trotzdem stieg er ein. Im Zug war es vollkommen still. Keine anderen Fahrg ste waren zu sehen. Daniel setzte sich ans Fenster und beobachtete die Stadt, die langsam hinter ihm verschwand. Nach ungef hr zehn Minuten bemerkte er etwas Merkw rdiges. Der Zug fuhr nicht mehr durch die Stadt. Drau en sah er pl tzlich einen kleinen Ort, den er kannte. Ein Haus stand am Ende einer Stra e. Daniel erkannte es sofort. Es war das Haus seiner Gro mutter. Seine Gro mutter war vor acht Jahren gestorben. Daniel sprang auf. Das kann nicht sein. Der Zug hielt. Die T ren ffneten sich. Vor dem Haus stand ein kleines M dchen. Daniel erkannte sie nach einigen Sekunden. Es war er selb

# création d'une Classe pour la pipeline Word

In [3]:
class WordIngestion:
    def __init__(self, chunk_size=1000, chunk_overlap=150):
        self.splitter=RecursiveCharacterTextSplitter(
            chunk_size= chunk_size,
            chunk_overlap= chunk_overlap,
            separators=["\n\n","\n",".",","]
        )
    def _clean(self,text:str)-> str:
        return clean_word_text  
    def process(self,path:str):
        documents= Docx2txtLoader(path).load()
        for d in documents:
            d.page_content= self._clean(d.page_content)
        return self.splitter.split_documents(documents)      

# Application

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def clean_word_text(text: str) -> str:
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

class WordIngestion:
    def __init__(self, chunk_size=900, chunk_overlap=120):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", ","]
        )

    def _clean(self, text: str) -> str:
        return clean_word_text(text)

    def process(self, path: str):
        documents = Docx2txtLoader(path).load()
        for doc in documents:
            doc.page_content = self._clean(doc.page_content)
        return self.splitter.split_documents(documents)

ingestion_word = WordIngestion(
    chunk_size=900,
    chunk_overlap=120
)

NameError: name 'RecursiveCharacterTextSplitter' is not defined

In [ ]:
chunk = inges